In [8]:
import torch

from usta_model import UstaModel
from usta_tokenizer import UstaTokenizer

device = "cpu"

if torch.cuda.is_available():
  device = "cuda"
elif torch.backends.mps.is_available():
  device = "mps"
  

print(f"Using device: {device}")

u_tokenizer = UstaTokenizer("tokenizer.json")

prompts = [
  "the capital of the united",
  "madrid is in",
  "the capital of france is",
  "the capital of germany is"
]

tokens = u_tokenizer.encode(prompts[0])
tokens = tokens.to(device)
print(tokens)
batch_tokens = u_tokenizer.encode_batch(prompts, 32)
batch_tokens = batch_tokens.to(device)
batch_tokens.shape

Using device: mps
tensor([ 0, 61,  1, 61,  2, 61,  0, 61,  3], device='mps:0')


torch.Size([4, 32])

In [9]:
torch.manual_seed(1)
context_length = 32
torch.manual_seed(1)
u_model = UstaModel(
  vocab_size=len(u_tokenizer.vocab),
  embedding_dim=12,
  num_heads=4,
  context_length=context_length,
  num_layers=8,
  device=device
)

In [10]:
out = u_model(batch_tokens)
out.shape

torch.Size([4, 32, 64])

In [11]:
out.flatten(0, 1).shape

torch.Size([128, 64])

In [15]:
with open("text.txt", "r") as f:
  text = f.read()

len(text), text[:100]

(4101,
 'the capital of the united states is not london. the capital of france is paris, and berlin is the ca')

In [16]:
token_ids = u_tokenizer.encode(text)
len(token_ids), type(token_ids)

(1593, torch.Tensor)

In [17]:
from text_dataset import create_data_loader

stride = 12

In [18]:
train_data_loader = create_data_loader(token_ids.tolist(), context_length, stride, 16, False)

len(train_data_loader)

9

In [19]:
# model parameters count
parameters_count = sum(p.numel() for p in u_model.parameters())
print(parameters_count)

# model architecture
print(u_model)

11776
UstaModel(
  (embedding): UstaEmbedding(
    (embedding): Embedding(64, 12)
  )
  (layers): Sequential(
    (0): UstaDecoderBlock(
      (self_attention): UstaMultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=12, out_features=12, bias=True)
        )
        (projection): Linear(in_features=12, out_features=12, bias=True)
      )
      (norm1): UstaLayerNorm()
      (mlp): UstaMLP(
        (gate_proj): Linear(in_features=12, out_features=12, bias=True)
        (up_proj): Linear(in_features=12, out_features=12, bias=True)
        (down_proj): Linear(in_features=12, out_features=12, bias=True)
        (gelu): GELU()
      )
      (norm2): UstaLayerNorm()
    )
    (1): UstaDecoderBlock(
      (self_attention): UstaMultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=12, out_features=12, bias=True)
        )
    

In [20]:
import torch.nn as nn

loss_fn = nn.CrossEntropyLoss()

In [21]:
# optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)
optimizer = torch.optim.AdamW(u_model.parameters(), lr=1e-3)

In [22]:
for i, (X, Y) in enumerate(train_data_loader):
  print(X.shape, Y.shape, Y.flatten().shape)
  break

torch.Size([16, 32]) torch.Size([16, 32]) torch.Size([512])


In [35]:
epoch = 9000

for epoch in range(epoch):
  total_loss = 0.
  for i, (X, Y) in enumerate(train_data_loader):
    X = X.to(device)
    Y = Y.to(device)
    
    pred = u_model(X)
    loss = loss_fn(pred.flatten(0, 1), Y.flatten())
    total_loss += loss.item()
    
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    
  average_loss = total_loss / len(train_data_loader)
  print(f"Epoch {epoch + 1} loss: {loss.item()} average loss: {average_loss}")

Epoch 1 loss: 0.38143065571784973 average loss: 0.5066428482532501
Epoch 2 loss: 0.37049099802970886 average loss: 0.5237099329630533
Epoch 3 loss: 0.3665136992931366 average loss: 0.5067573123508029
Epoch 4 loss: 0.3802768886089325 average loss: 0.5204738974571228
Epoch 5 loss: 0.3514308035373688 average loss: 0.5034509433640374
Epoch 6 loss: 0.402031272649765 average loss: 0.5043050315645006
Epoch 7 loss: 0.38732588291168213 average loss: 0.5043893324004279
Epoch 8 loss: 0.37475791573524475 average loss: 0.5044713881280687
Epoch 9 loss: 0.3569948375225067 average loss: 0.5099783970250024
Epoch 10 loss: 0.3756103217601776 average loss: 0.5201242102517022
Epoch 11 loss: 0.37671735882759094 average loss: 0.5063543617725372
Epoch 12 loss: 0.41943493485450745 average loss: 0.5202583207024468
Epoch 13 loss: 0.3653999865055084 average loss: 0.498960898982154
Epoch 14 loss: 0.2976541519165039 average loss: 0.49973448117574054
Epoch 15 loss: 0.3515804708003998 average loss: 0.504242350657781


In [36]:
import torch

new_tokens = u_tokenizer.encode("the capital of the united states is")
new_tokens = new_tokens.tolist()
# new_tokens.append(61)

out = u_model(torch.tensor([new_tokens]).to(device))
out = out.squeeze(0)
probs = torch.softmax(out[-1], dim=-1)
max_prob, max_index = torch.max(probs, dim=-1)
max_prob, max_index, probs

(tensor(0.6576, device='mps:0', grad_fn=<MaxBackward0>),
 tensor(60, device='mps:0'),
 tensor([1.2077e-07, 7.9721e-12, 8.5738e-15, 1.8298e-16, 7.4554e-17, 6.1674e-14,
         3.6320e-10, 3.2107e-14, 5.9758e-13, 4.0177e-14, 1.6526e-18, 1.1679e-15,
         3.0616e-22, 2.2816e-10, 3.0719e-14, 1.8672e-18, 1.6027e-16, 3.8123e-13,
         3.8676e-13, 2.6389e-05, 1.1935e-14, 6.4315e-16, 2.2731e-19, 1.1805e-12,
         9.5605e-09, 1.0440e-15, 5.4642e-17, 4.8046e-15, 3.8848e-15, 3.3728e-13,
         3.1619e-12, 2.4321e-17, 1.9056e-08, 1.0076e-10, 3.0646e-19, 4.6971e-12,
         2.3937e-23, 3.7526e-12, 1.3778e-05, 4.0306e-12, 3.3596e-11, 3.1683e-14,
         2.6900e-13, 3.9817e-16, 3.2695e-14, 1.0696e-07, 2.2880e-13, 5.0264e-14,
         1.9562e-09, 1.8306e-19, 1.3278e-15, 9.7501e-12, 3.4726e-09, 5.0038e-21,
         3.8655e-18, 3.6538e-14, 3.1058e-14, 6.7609e-07, 1.5055e-08, 8.5176e-03,
         6.5758e-01, 3.3386e-01, 1.2393e-13, 1.2395e-13], device='mps:0',
        grad_fn=<SoftmaxBackwa

In [40]:
# save model
torch.save(u_model.state_dict(), "u_model.pth")

# load model
u_model.load_state_dict(torch.load("u_model.pth"))

# generate text
new_tokens = u_tokenizer.encode("the capital of the united states is london. the capital of france is")
new_tokens = new_tokens.detach().cpu().numpy().tolist()
new_tokens.append(61)
len(new_tokens)

28

In [38]:
loaded_model = UstaModel(64, embedding_dim=12, num_heads=4, context_length=32, num_layers=8, device=device)
loaded_model.load_state_dict(torch.load("u_model.pth"))
loaded_model

UstaModel(
  (embedding): UstaEmbedding(
    (embedding): Embedding(64, 12)
  )
  (layers): Sequential(
    (0): UstaDecoderBlock(
      (self_attention): UstaMultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=12, out_features=12, bias=True)
        )
        (projection): Linear(in_features=12, out_features=12, bias=True)
      )
      (norm1): UstaLayerNorm()
      (mlp): UstaMLP(
        (gate_proj): Linear(in_features=12, out_features=12, bias=True)
        (up_proj): Linear(in_features=12, out_features=12, bias=True)
        (down_proj): Linear(in_features=12, out_features=12, bias=True)
        (gelu): GELU()
      )
      (norm2): UstaLayerNorm()
    )
    (1): UstaDecoderBlock(
      (self_attention): UstaMultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=12, out_features=12, bias=True)
        )
        (p

In [39]:
out = u_model(torch.tensor(new_tokens).unsqueeze(0).to(device))

probs = torch.softmax(out[-1], dim=-1)
max_prob, max_index = torch.max(probs, dim=-1)
max_prob, max_index, probs

(tensor([1.0000, 0.1860, 1.0000, 0.5437, 0.9909, 0.6596, 0.9999, 0.4951, 1.0000,
         0.1940, 0.9540, 0.9682, 0.2367, 1.0000, 0.3633, 1.0000, 1.0000, 0.4113,
         1.0000, 0.3117, 1.0000, 0.2639, 1.0000, 0.4139, 0.9997, 0.2360, 1.0000,
         0.4844], device='mps:0', grad_fn=<MaxBackward0>),
 tensor([61, 41, 61,  1, 61, 40, 61, 41, 61,  1, 58, 61, 43, 61,  0, 61, 61, 37,
         61, 10, 61, 48, 61,  1, 61, 41, 61,  9], device='mps:0'),
 tensor([[1.9332e-12, 7.1746e-15, 3.7926e-22,  ..., 1.0000e+00, 8.8815e-17,
          8.8835e-17],
         [5.8635e-04, 1.3817e-02, 1.2974e-01,  ..., 8.0209e-12, 6.9240e-09,
          6.9241e-09],
         [1.0847e-12, 1.8504e-15, 4.4814e-22,  ..., 1.0000e+00, 2.4060e-16,
          2.4065e-16],
         ...,
         [7.8819e-02, 3.1055e-03, 2.0028e-01,  ..., 1.9577e-16, 4.6133e-10,
          4.6132e-10],
         [1.4702e-12, 4.4309e-19, 6.4585e-26,  ..., 1.0000e+00, 7.4598e-19,
          7.4625e-19],
         [1.0331e-02, 8.8479e-02, 2.2342e